# 03 — Model Training

Trains a LightGBM model on `data/processed/laps_features.csv` and saves it to `data/processed/model.pkl`.

**Run order:**
1. Train model
2. Evaluate metrics
3. Feature importance
4. Prediction vs actual plots
5. Per-compound and per-track error analysis

In [ ]:
import sys, os
sys.path.append("..")

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")

import importlib
import src.model as mdl
importlib.reload(mdl)
from src.model import train, evaluate, feature_importance, load_model, make_split
from src.features import FEATURE_COLUMNS, TARGET_COLUMN

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
PROC_DIR = os.path.join(BASE_DIR, "data", "processed")

pd.set_option("display.float_format", "{:.3f}".format)
print("Imports OK")

## 1. Load features

In [ ]:
df = pd.read_csv(os.path.join(PROC_DIR, "laps_features.csv"))
print(f"Loaded {len(df):,} laps")
print(f"Features available: {[c for c in FEATURE_COLUMNS if c in df.columns]}")
missing = [c for c in FEATURE_COLUMNS if c not in df.columns]
if missing:
    print(f"MISSING features: {missing}")

## 2. Train

Early stopping monitors MAE on the test set every round and stops if no improvement for 50 rounds.

In [ ]:
model = train(df)

## 3. Evaluate

In [ ]:
results = evaluate(model, df)

print(f"Train MAE  : {results['train_mae']:.3f}s")
print(f"Train RMSE : {results['train_rmse']:.3f}s")
print(f"Test  MAE  : {results['test_mae']:.3f}s")
print(f"Test  RMSE : {results['test_rmse']:.3f}s")
print()
print("Target is RelativePace = seconds above/below event median lap time.")
print("A MAE of 0.3s means the model misjudges a driver's relative pace by ~0.3s/lap.")
print("Absolute lap time = FP2LongRunPace + PredictedRelativePace.")

In [ ]:
test_df = results["test_df"].copy()
test_df["Predicted"] = results["test_preds"]
test_df["Error"]     = test_df["Predicted"] - test_df[TARGET_COLUMN]

sample = test_df.sample(min(3000, len(test_df)), random_state=42)

fig = px.scatter(
    sample,
    x=TARGET_COLUMN,
    y="Predicted",
    color="Compound",
    color_discrete_map={"SOFT": "red", "MEDIUM": "yellow", "HARD": "white"},
    opacity=0.5,
    title="Predicted vs Actual RelativePace (test set — 2025 R19+)",
    labels={TARGET_COLUMN: "Actual RelativePace (s)", "Predicted": "Predicted RelativePace (s)"},
    height=500,
)

mn = sample[TARGET_COLUMN].min()
mx = sample[TARGET_COLUMN].max()
fig.add_shape(type="line", x0=mn, y0=mn, x1=mx, y1=mx,
              line=dict(color="white", dash="dash"))
fig.show()

In [ ]:
# Error distribution — should be centred at 0
fig = px.histogram(
    test_df,
    x="Error",
    nbins=80,
    title="Prediction error distribution (test set)",
    labels={"Error": "Error (s) — Predicted minus Actual"},
    height=400,
)
fig.add_vline(x=0, line_color="red", line_dash="dash")
fig.show()

## 4. Feature importance

**Gain** = how much each feature reduces prediction error when it is used for a split.
Higher = more important.

In [ ]:
imp = feature_importance(model)
print(imp.to_string(index=False))

In [ ]:
fig = px.bar(
    imp,
    x="Importance",
    y="Feature",
    orientation="h",
    title="Feature Importance (split count)",
    height=650,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

### Gain importance

**Split count** = how many times the feature was used to split.  
**Gain** = how much each split actually reduced prediction error. More meaningful — a feature used 50 times with big error reductions beats one used 300 times with tiny ones.

In [ ]:
gain_imp = pd.DataFrame({
    "Feature":    FEATURE_COLUMNS,
    "Gain":       model.booster_.feature_importance(importance_type="gain"),
}).sort_values("Gain", ascending=False).reset_index(drop=True)

print(gain_imp.to_string(index=False))

fig = px.bar(
    gain_imp,
    x="Gain",
    y="Feature",
    orientation="h",
    title="Feature Importance (gain — error reduction per split)",
    height=550,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

## 5. Per-compound error

In [ ]:
compound_err = (
    test_df.groupby("Compound")["Error"]
    .agg(MAE=lambda x: x.abs().mean(), RMSE=lambda x: np.sqrt((x**2).mean()), Count="count")
    .reset_index()
    .sort_values("MAE")
)
print(compound_err.to_string(index=False))

## 6. Per-track error

In [ ]:
track_err = (
    test_df.groupby("EventName")["Error"]
    .agg(MAE=lambda x: x.abs().mean(), RMSE=lambda x: np.sqrt((x**2).mean()), Count="count")
    .reset_index()
    .sort_values("MAE")
)

fig = px.bar(
    track_err,
    x="MAE",
    y="EventName",
    orientation="h",
    title="MAE per track (test set)",
    labels={"MAE": "Mean Absolute Error (s)"},
    height=500,
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

## 7. Tyre degradation curve — sanity check

For one driver at one race, compare predicted vs actual lap time as tyres wear.
Should show an upward curve as tyre life increases.

In [ ]:
# Pick the first test race and a front-runner
test_rounds = sorted(test_df["RoundNumber"].unique())
sample_round = test_rounds[0]
sample_event = test_df[test_df["RoundNumber"] == sample_round]["EventName"].iloc[0]

# Pick the driver with the most laps in this race
race_laps = test_df[test_df["EventName"] == sample_event].copy()
top_driver = race_laps.groupby("Driver")["LapNumber"].count().idxmax()
driver_laps = race_laps[race_laps["Driver"] == top_driver].sort_values("LapNumber")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=driver_laps["LapNumber"], y=driver_laps[TARGET_COLUMN],
    mode="markers+lines", name="Actual", line=dict(color="cyan")
))
fig.add_trace(go.Scatter(
    x=driver_laps["LapNumber"], y=driver_laps["Predicted"],
    mode="markers+lines", name="Predicted", line=dict(color="orange", dash="dash")
))
fig.update_layout(
    title=f"{top_driver} — {sample_event} (predicted vs actual)",
    xaxis_title="Lap Number",
    yaxis_title="Lap Time (s)",
    height=450,
)
fig.show()